# 后台标定与最终发布

独立合成 sandbox，不连接设备，也不依赖前面课程的数据。源码在 `src/my_experiment/task_calibration.py`；模拟模型和联合判定在旁边的 `joint_calibration.py`。

本课分清三件事：**拟合输入可用、组合参数通过验证、分支已发布**。它们不是同一个成功状态。先保留两条通道的候选，再由一个收尾 procedure 重新测量组合结果并决定是否发布。

In [ ]:
import scopecat as sc

session = sc.notebook()
session

In [ ]:
from uuid import uuid4

from my_experiment.calibration import Sensor
from my_experiment.task_calibration import FinishIntent, create_task

assert session.project_root is not None
project = sc.open_project(session.project_root)
lab = project.connect()
trial = "background-" + uuid4().hex[:8]
initial = lab.parameters.save(
    name=trial + "-initial",
    catalog=sc.parameter_catalog("sensors", Sensor),
    parameters=sc.parameter_snapshot(
        "inputs",
        tables={
            Sensor: [
                Sensor(id="q0", offset=0.1),
                Sensor(id="q1", offset=0.1),
            ]
        },
    ),
)
destinations = {}
for case, coupling in (("accepted", 0.0), ("rejected", 1.0), ("conflict", 0.0)):
    name = trial + "-" + case
    head = lab.parameters.create_branch(name, revision=initial)
    destinations[case] = head
    create_task(
        lab,
        name,
        FinishIntent(
            initial=lab.parameters.resolve(initial),
            destination=head,
            revision_name=name + "-result",
            coupling=coupling,
        ),
    )
# 模拟另一位操作者在任务捕获分支后保存参数。值不变也会推进 generation。
concurrent = lab.parameters.checkout(trial + "-conflict").save(
    catalog=initial.catalog,
    parameters=initial.parameters,
    note="并发维护",
)
task_ids = [trial + "-" + case for case in destinations]
print(task_ids)
lab.close()

## 交给后台，Notebook 可以断开

任务定义和分支基线已经保存在 daemon。下一格重新连接并启动任务；之后关闭 Notebook 不会取消工作。重新打开时可用 `lab.calibration_tasks.list()` 找回任务，也可在工作台 Calibration tasks 查看。无需手写 JSON 文件保存 id。

`fit_stage` 与 `finalize` 已注册在本 sandbox 的代码目录中，因此后台 worker 可以加载它们。

In [ ]:
lab = project.connect()
for task_id in task_ids:
    task = lab.calibration_tasks.get(task_id)
    lab.calibration_tasks.start(task, actor="learner", reason="体验后台完整标定")
print("已提交后台；可到工作台查看阶段和收尾执行。")

## 等待仅用于本课观察

下面的轮询不执行实验，也不负责推进任务。超时只表示还没完成，可以查看工作台诊断后重新运行这一格。真实工作中不必让 Notebook 等着。

Pause/Cancel 停止新的阶段准入；已经提交的 procedure 需要在其执行页面单独检查或取消。

In [ ]:
import time

deadline = time.monotonic() + 90
while True:
    views = {
        case: lab.calibration_tasks.get(trial + "-" + case) for case in destinations
    }
    if all(view.task.mode == "finished" for view in views.values()):
        break
    if time.monotonic() >= deadline:
        raise TimeoutError("后台尚未结束；请查看任务和 procedure 的状态，再运行本格")
    time.sleep(0.25)
for case, view in views.items():
    final = view.finalization
    print(
        case,
        "checks:",
        view.progress.successful,
        "final:",
        final.closure if final else view.task.finalization_error,
    )

In [ ]:
from my_experiment.joint_calibration import JOINT_DECISION

from scopecat.automation import (
    AnalysisPublicationOutputRef,
    ParameterBranchPublishOutputRef,
)

assert all(view.progress.successful for view in views.values())
accepted = lab.parameters.checkout(trial + "-accepted").head
assert accepted.generation == destinations["accepted"].generation + 1
assert lab.parameters.checkout(trial + "-rejected").head == destinations["rejected"]
assert lab.parameters.checkout(trial + "-conflict").head == concurrent
for case, view in views.items():
    assert view.finalization is not None
    procedure = lab.procedures.get(view.finalization.procedure_run_id)
    print(case, procedure.summary())
    verification = procedure.output("verify-joint")
    assert isinstance(verification, AnalysisPublicationOutputRef)
    decision = lab.published_analysis(verification.analysis_record_id).fact_as(
        "decision", JOINT_DECISION
    )
    assert decision.checked == ("q0", "q1") and not decision.missing
    assert decision.accepted == (case != "rejected")
    if case == "accepted":
        receipt = procedure.output("publish")
        assert isinstance(receipt, ParameterBranchPublishOutputRef)
        assert receipt.branch == accepted
    else:
        assert procedure.summary().outcome == "failed"
        if case == "conflict":
            assert procedure.step("publish").state == "failed"
            assert "branch changed" in (procedure.summary().reason or "")
        else:
            assert not any(
                step.operation == "parameter_publish"
                for step in procedure.steps().items
            )
print("只有 accepted 分支被本轮标定推进。")

## 看证据，再改模型

三个任务的阶段检查都通过，但 rejected 的合成耦合项使组合结果超出容差；conflict 的最终测量可以通过，但捕获的分支 generation 已过期，不能把旧证据自动套到新分支上。`finished` 只表示任务推进结束，不等于已发布。

打开收尾执行：比较 `compose`、`verify-q0/q1`、`verify-joint` 和 `publish`。失败不会抹掉已经取得的候选或测量。

可尝试调整耦合强度或容差后从创建新任务的格子开始重跑。不要更改现有任务的定义或在重试时换用最新分支头。本课先用同基线候选；顺序候选需让后一步确实消费前一步的候选，再以 `mode="sequential"` 汇总。

In [ ]:
lab.close()